<a href="https://colab.research.google.com/github/svkamat111922-sketch/Online-Store-Inventory/blob/main/Underwater_Image_Enhancement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files

# Upload BOTH:
# 1. images.zip (your 950 images)
# 2. hybrid_underwater_dataset_updated.xlsx
uploaded = files.upload()

Saving images.zip to images.zip


In [ ]:
from google.colab import files

# Upload BOTH:
# 1. images.zip (your 950 images)
# 2. hybrid_underwater_dataset_updated.xlsx
uploaded = files.upload()

Saving hybrid_underwater_dataset_updated.xlsx to hybrid_underwater_dataset_updated.xlsx


In [ ]:
import zipfile
import os

ZIP_PATH = "images.zip"   # make sure name matches upload
EXTRACT_DIR = "/content/images"

os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("✅ Extracted files:", len(os.listdir(EXTRACT_DIR)))

✅ Extracted files: 950


In [ ]:
import pandas as pd

EXCEL_PATH = "hybrid_underwater_dataset_updated.xlsx"

df = pd.read_excel(EXCEL_PATH)

print("Total rows in Excel:", len(df))
print(df.head())

Total rows in Excel: 950
   Image_ID Dataset_Source Water_Type          Lighting Color_Cast Resolution  \
0  IMG_0001      Synthetic   Deep Sea            Bright      Green    512x512   
1  IMG_0002           EUVP   Deep Sea  Artificial Light      Green    512x512   
2  IMG_0003           RUIE      Clear  Artificial Light      Green  1024x1024   
3  IMG_0004           EUVP      Murky            Bright       Blue    256x256   
4  IMG_0005           UIEB     Turbid         Low Light       Blue    256x256   

        Augmentation  PSNR_dB   SSIM  UIQM Object_Type  
0  Contrast Enhanced    28.57  0.926  4.55   Submarine  
1               Blur    27.69  0.885  4.28       Cable  
2                NaN    27.86  0.945  4.42        Mine  
3                NaN    30.40  0.892  4.39   Submarine  
4                NaN    27.00  0.843  4.72   Submarine  


In [ ]:
MARITIME_CLASSES = [
    "Submarine", "Mine", "Cable", "Vessel", "Underwater Structure"
]

# Clean column values
df["Object_Type"] = df["Object_Type"].astype(str).str.strip()

# Filter
df_filtered = df[df["Object_Type"].isin(MARITIME_CLASSES)]

print("After filtering:", len(df_filtered))

# If 0 → fallback (VERY IMPORTANT)
if len(df_filtered) == 0:
    print("⚠️ No maritime labels found → using full dataset")
    df_filtered = df

After filtering: 950


In [ ]:
import cv2
import numpy as np
import random

def add_blur(image):
    k = random.choice([3,5,7])
    return cv2.GaussianBlur(image, (k, k), 0)

def add_color_cast(image):
    img = image.astype(np.float32)
    img[:,:,2] *= random.uniform(0.5, 0.8)  # Red ↓
    img[:,:,1] *= random.uniform(0.9, 1.1)  # Green
    img[:,:,0] *= random.uniform(1.0, 1.3)  # Blue ↑
    return np.clip(img, 0, 255).astype(np.uint8)

def reduce_contrast(image):
    alpha = random.uniform(0.5, 0.8)
    beta = random.uniform(-20, 10)
    return cv2.convertScaleAbs(image, alpha=alpha, beta=beta)

def simulate_underwater(image):
    img = add_blur(image)
    img = add_color_cast(img)
    img = reduce_contrast(img)
    return img

In [ ]:
def augment_pair(imgA, imgB):
    if random.random() > 0.5:
        imgA = cv2.flip(imgA, 1)
        imgB = cv2.flip(imgB, 1)

    if random.random() > 0.5:
        imgA = cv2.flip(imgA, 0)
        imgB = cv2.flip(imgB, 0)

    if random.random() > 0.5:
        k = random.choice([1,2,3])
        imgA = np.rot90(imgA, k)
        imgB = np.rot90(imgB, k)

    return imgA.copy(), imgB.copy()

In [ ]:
OUTPUT_DIR = "/content/dataset"
TRAIN_A = os.path.join(OUTPUT_DIR, "trainA")
TRAIN_B = os.path.join(OUTPUT_DIR, "trainB")

os.makedirs(TRAIN_A, exist_ok=True)
os.makedirs(TRAIN_B, exist_ok=True)

image_files = sorted(os.listdir(EXTRACT_DIR))

print("Total images found:", len(image_files))

count = 0

for img_name in image_files:

    # Ensure only image files are used
    if not img_name.lower().endswith((".jpg", ".png", ".jpeg")):
        continue

    img_path = os.path.join(EXTRACT_DIR, img_name)

    clean = cv2.imread(img_path)

    if clean is None:
        print("❌ Corrupt:", img_name)
        continue

    # Resize for U-Net
    clean = cv2.resize(clean, (256, 256))

    # Create degraded version
    degraded = simulate_underwater(clean)

    # Apply augmentation AFTER pairing
    degraded, clean = augment_pair(degraded, clean)

    # KEEP SAME NAME for pairing
    filename = img_name   # 🔥 IMPORTANT

    cv2.imwrite(os.path.join(TRAIN_A, filename), degraded)
    cv2.imwrite(os.path.join(TRAIN_B, filename), clean)

    count += 1

print("✅ Total paired images created:", count)

Total images found: 950
✅ Total paired images created: 950


In [ ]:
import shutil

shutil.make_archive("/content/maritime_dataset", 'zip', OUTPUT_DIR)

'/content/maritime_dataset.zip'